### Weather Dataset 

In [2]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
from rapidfuzz import process 

### Importing Dataset 

In [3]:
df = pd.read_csv('weather_data.csv')

### Copy the Dataset 

In [4]:
weather_copy = df.copy()

### Basic Inspection

In [5]:
weather_copy.head()

,weather_id,city,datetime,temperature,rainfall_mm,humidity,visibility_km,wind_speed,weather_type
0,W1,DELHI,Jul 26 2024 07:12 PM,26.0,103.95,99.8,11.3,13.1,NaN
1,W2,Hyderabad,22/08/2024 12:41,28.4,1.96,45.6,3.5,31.7,Rain
2,W3,Pune,26/12/2024 14:36,37.4,67.65,69.6,9.7,28.0,Clear
3,W4,delhi,09-20-2025 11:35,19.3,70.03,78.8,12.7,12.0,Storm
4,W5,delhi,2025-07-09 23:33:18,29.4,91.78,97.6,5.2,2.7,Cloudy


In [6]:
weather_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   weather_id     80000 non-null  str    
 1   city           80000 non-null  str    
 2   datetime       80000 non-null  str    
 3   temperature    80000 non-null  float64
 4   rainfall_mm    80000 non-null  float64
 5   humidity       80000 non-null  float64
 6   visibility_km  74420 non-null  float64
 7   wind_speed     80000 non-null  float64
 8   weather_type   66664 non-null  str    
dtypes: float64(5), str(4)
memory usage: 5.5 MB


In [7]:
weather_copy.describe()

,temperature,rainfall_mm,humidity,visibility_km,wind_speed
count,80000.000000,80000.000000,80000.000000,74420.000000,80000.000000
mean,30.021030,75.052630,69.975463,7.758289,22.981419
std,6.931953,43.350091,17.291429,4.180337,12.719365
min,18.000000,0.000000,40.000000,0.500000,1.000000
25%,24.000000,37.567500,55.000000,4.100000,12.000000
50%,30.000000,74.840000,70.100000,7.800000,23.000000
75%,36.000000,112.392500,84.900000,11.400000,34.000000
max,42.000000,150.000000,100.000000,15.000000,45.000000


In [8]:
weather_copy.isnull().sum()

weather_id           0
city                 0
datetime             0
temperature          0
rainfall_mm          0
humidity             0
visibility_km     5580
wind_speed           0
weather_type     13336
dtype: int64

In [9]:
weather_copy.duplicated().sum()


np.int64(0)

In [10]:
weather_copy['city'].value_counts()


city
Hyderbad     4873
mumbai       4796
delhi        4767
DELHI        4758
MUMBAI       4755
PUNE         4742
Bangalore    4735
Mubmai       4722
Pune         4704
Banglore     4681
bengaluru    4674
Hyderabad    4648
Bombay       4646
Mumbai       4644
Delhi        4634
HYD          4616
pune         4605
Name: count, dtype: int64

In [11]:
valid_cities = ['mumbai', 'delhi', 'bangalore', 'pune', 'hyderabad']


def correct_city_name(city):

    match = process.extractOne(city, valid_cities)

    return match[0]


weather_copy['city'] = weather_copy['city'].apply(correct_city_name)

In [12]:
weather_copy['datetime'] = pd.to_datetime(
    weather_copy['datetime'], 
    format = 'mixed',
    errors='coerce'
)

### Check Failed Date Conversions

In [13]:
print(weather_copy[weather_copy['datetime'].isnull()])

Empty DataFrame
Columns: [weather_id, city, datetime, temperature, rainfall_mm, humidity, visibility_km, wind_speed, weather_type]
Index: []


In [14]:
weather_copy.isnull().sum()

weather_id           0
city                 0
datetime             0
temperature          0
rainfall_mm          0
humidity             0
visibility_km     5580
wind_speed           0
weather_type     13336
dtype: int64

In [15]:
weather_copy.duplicated().sum()

np.int64(0)

In [16]:
weather_copy.describe()

,datetime,temperature,rainfall_mm,humidity,visibility_km,wind_speed
count,80000,80000.000000,80000.000000,80000.000000,74420.000000,80000.000000
mean,2025-03-03 16:38:58.596187,30.021030,75.052630,69.975463,7.758289,22.981419
min,2024-01-01 00:08:00,18.000000,0.000000,40.000000,0.500000,1.000000
25%,2024-07-31 19:43:26.250000,24.000000,37.567500,55.000000,4.100000,12.000000
50%,2025-03-02 15:22:30,30.000000,74.840000,70.100000,7.800000,23.000000
75%,2025-09-30 03:18:15.750000,36.000000,112.392500,84.900000,11.400000,34.000000
max,2026-12-04 23:54:00,42.000000,150.000000,100.000000,15.000000,45.000000
std,NaN,6.931953,43.350091,17.291429,4.180337,12.719365


In [17]:
weather_copy['weather_type'].value_counts()

weather_type
Storm     13415
Clear     13396
Fog       13386
Rain      13384
Cloudy    13083
Name: count, dtype: int64

In [18]:
weather_copy['weather_type'] = weather_copy['weather_type'].str.lower()

### Handling visibility_km

In [19]:
weather_copy["visibility_km"] = (
    weather_copy["visibility_km"]
    .fillna(
        weather_copy["visibility_km"].median()
    )
)

### Handling weather_type

In [20]:
weather_copy["weather_type"] = (
    weather_copy["weather_type"]
    .fillna(
        weather_copy["weather_type"].mode()[0]
    )
)

### Create Weather Severity Flag ⭐

In [21]:
weather_copy["weather_severity"] = np.where(
    (
        (weather_copy["rainfall_mm"] > 50)
        |
        (weather_copy["wind_speed"] > 40)
        |
        (weather_copy["visibility_km"] < 2)
    ),
    "Severe",
    "Normal"
)

### Extract Time Features

In [22]:
weather_copy["year"] = (
    weather_copy["datetime"].dt.year
)

weather_copy["month"] = (
    weather_copy["datetime"].dt.month_name()
)

weather_copy["day"] = (
    weather_copy["datetime"].dt.day_name()
)

weather_copy["hour"] = (
    weather_copy["datetime"].dt.hour
)

### Final Validation

In [23]:
weather_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   weather_id        80000 non-null  str           
 1   city              80000 non-null  str           
 2   datetime          80000 non-null  datetime64[us]
 3   temperature       80000 non-null  float64       
 4   rainfall_mm       80000 non-null  float64       
 5   humidity          80000 non-null  float64       
 6   visibility_km     80000 non-null  float64       
 7   wind_speed        80000 non-null  float64       
 8   weather_type      80000 non-null  str           
 9   weather_severity  80000 non-null  str           
 10  year              80000 non-null  int32         
 11  month             80000 non-null  str           
 12  day               80000 non-null  str           
 13  hour              80000 non-null  int32         
dtypes: datetime64[us](1), float64(5),

In [24]:
weather_copy.isnull().sum()

weather_id          0
city                0
datetime            0
temperature         0
rainfall_mm         0
humidity            0
visibility_km       0
wind_speed          0
weather_type        0
weather_severity    0
year                0
month               0
day                 0
hour                0
dtype: int64

In [25]:
weather_copy.shape

(80000, 14)

In [26]:
weather_copy.head()

,weather_id,city,datetime,temperature,rainfall_mm,humidity,visibility_km,wind_speed,weather_type,weather_severity,year,month,day,hour
0,W1,mumbai,2024-07-26 19:12:00,26.0,103.95,99.8,11.3,13.1,storm,Severe,2024,July,Friday,19
1,W2,hyderabad,2024-08-22 12:41:00,28.4,1.96,45.6,3.5,31.7,rain,Normal,2024,August,Thursday,12
2,W3,pune,2024-12-26 14:36:00,37.4,67.65,69.6,9.7,28.0,clear,Severe,2024,December,Thursday,14
3,W4,delhi,2025-09-20 11:35:00,19.3,70.03,78.8,12.7,12.0,storm,Severe,2025,September,Saturday,11
4,W5,delhi,2025-07-09 23:33:18,29.4,91.78,97.6,5.2,2.7,cloudy,Severe,2025,July,Wednesday,23


### Save Final Cleaned Weather Dimension

In [27]:
weather_copy.to_csv(
    "dim_weather_data.csv",
    index=False
)

### Create City Dimension

In [28]:
city_df = pd.DataFrame({

    "city_id": [1,2,3,4,5],

    "city": [
        "mumbai",
        "delhi",
        "bangalore",
        "hyderabad",
        "pune"
    ]
})

### Merge into Weather Table


In [29]:
weather_copy = weather_copy.merge(
    city_df,
    on="city",
    how="left"
)

### Saving after resolving Data Modelling Issue

In [30]:
weather_copy.to_csv(
    "dims_weather_data.csv",
    index=False
)